# Princeton, New Jersey — LVT Policy Analysis

**Princeton** is a consolidated municipality in **Mercer County, NJ**, formed in
January 2013 when Princeton Borough and Princeton Township merged into a single
municipality (tax district code **1114**). Home to Princeton University, it is a
small, high-value, overwhelmingly residential town (~8,400 tax line items,
~31,000 residents).

## New Jersey property-tax mechanics

- NJ assesses real property at **true (market) value**, but each municipality
  carries a **director's / equalization ratio** (Princeton 2024 = **70.51%**)
  measuring assessed-to-true value. Because a revenue-neutral shift is computed
  *within* the municipality, this ratio scales land and improvements uniformly
  and **cancels** — the model runs on assessed values as filed in MOD-IV.
- The **general tax rate** ($2.663 / $100 in 2024) is the sum of five levies:
  county, county open space, local school, and the municipal-side levies. We
  model **only the municipal (city) levy** — NOT county or school.
- Tax rates are quoted **per $100** of assessed value; the model's millage is
  **per $1,000**, so `mills = rate_per_100 x 10`.

## Scope, reform, and the revenue target (this model)

| Parameter | Value |
|---|---|
| Reform | Split-rate property tax, **2:1** land:improvement |
| Scope | **City / municipal levy only** (excl. county, school) |
| Exemptions | **Preserve all** existing exemptions (class 15A-15F held out) |
| Assessment | Assessed values as filed (director's ratio 70.51%, cancels) |

**Official revenue figure (Gate 1 target): $40,100,104.00** — Princeton's
municipal levy = **Local Municipal Purpose Tax $36,735,248.96 + Municipal
Library Tax $3,364,855.04**. This is the "amount to be raised by taxation for
municipal purposes" in the town's 2024 adopted budget and corresponds to the
published municipal tax rate of **56.1 cents / $100** (Princeton 2024 Municipal
Budget Newsletter). Source: **2024 Abstract of Ratables for the County of
Mercer**, certified by the Mercer County Board of Taxation, Section 12-C(ii)
(Local Municipal Purposes), taxing district 14 (Princeton).

We **exclude** from scope: County tax ($55,628,474.93), County Open Space
($3,053,023.82), District School ($90,175,525.00), and the separately-levied
**Municipal Open Space** dedicated tax ($1,214,081.00). The five levies sum to
the certified total levy of $190,171,208.75 (arithmetic verified against the
Abstract).

## Data source

**Parcels and MOD-IV Composite of New Jersey** (statewide cadastral layer,
NJ Office of GIS / NJOGIS), served live at
`https://maps.nj.gov/arcgis/rest/services/Framework/Cadastral/MapServer/0`.
Each parcel carries the full MOD-IV attribute set inline — values
(`LAND_VAL`, `IMPRVT_VAL`, `NET_VALUE`, `LAST_YR_TX`), class/qualifier
(`PROP_CLASS`, `PCLQCODE`), the residential dwelling-unit count (`DWELL`) and
`COMM_DWELL`, building/use fields (`BLDG_CLASS`, `PROP_USE`, `LAND_DESC`),
sale fields (`SALE_PRICE`, `SALES_CODE`, `DEED_DATE`), and geometry. Owner
names are redacted per NJ's Daniel's Law. Filtered to **`PCL_MUN = '1114'`**
(consolidated Princeton).

In [ ]:
import sys
import json
import os
import time
from pathlib import Path

import geopandas as gpd
import matplotlib
matplotlib.use('Agg')  # headless
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
from shapely.geometry import Polygon, LinearRing
from dotenv import load_dotenv

sys.path.insert(0, '../..')
REPO_ROOT = Path('../..').resolve()
load_dotenv(REPO_ROOT / '.env')

from lvt.lvt_utils import (
    model_split_rate_tax,
    calculate_current_tax,
    calculate_category_tax_summary,
    print_category_tax_summary,
    save_standard_export,
)
from lvt.census_utils import get_census_data_with_boundaries, match_to_census_blockgroups
from lvt.viz import create_city_report
from lvt.parcel_map import save_parcel_map_export, create_parcel_map

# ---- Constants ----
CITY_NAME = 'princeton'
STATE_FIPS = '34'          # New Jersey
COUNTY_FIPS = '021'        # Mercer County
MODEL_TYPE = 'split_rate_2to1'
LAND_IMPROVEMENT_RATIO = 2.0

# Parcel-map identity columns
PARCEL_ID_COL = 'PAMS_PIN'
OWNER_NAME_COL = None          # redacted per NJ Daniel's Law
OWNER_ADDRESS_COL = None
PARCEL_URL_TEMPLATE = None     # NJ assessor lookups are session-based; no stable deep link

# ---- Official figures (2024 Mercer County Abstract of Ratables, district 14) ----
# Municipal levy modeled = Local Municipal Purpose + Municipal Library
OFFICIAL_MUNI_PURPOSE = 36_735_248.96
OFFICIAL_MUNI_LIBRARY = 3_364_855.04
OFFICIAL_REVENUE = OFFICIAL_MUNI_PURPOSE + OFFICIAL_MUNI_LIBRARY   # 40,100,104.00

# Certified assessed net valuation taxable = certified total levy / general tax rate
CERTIFIED_TOTAL_LEVY = 190_171_208.75
GENERAL_TAX_RATE_PER_100 = 2.663
CERTIFIED_NET_VAL = CERTIFIED_TOTAL_LEVY / (GENERAL_TAX_RATE_PER_100 / 100.0)  # ~$7.142B assessed

# Certified municipal (purpose + library) rate, per $100 and as millage (per $1,000)
MUNI_RATE_PER_100 = OFFICIAL_REVENUE / CERTIFIED_NET_VAL * 100.0
MUNI_MILLAGE = MUNI_RATE_PER_100 * 10.0

DIRECTORS_RATIO = 0.7051    # 2024 equalization ratio (informational; cancels in the shift)

DATA_DIR = Path('data')
DATA_DIR.mkdir(exist_ok=True)

print(f"Official municipal levy (target):  ${OFFICIAL_REVENUE:,.2f}")
print(f"Certified net valuation taxable:   ${CERTIFIED_NET_VAL:,.0f}")
print(f"Municipal rate:                    ${MUNI_RATE_PER_100:.5f}/$100  =  {MUNI_MILLAGE:.4f} mills")

## Section 2 — Fetch / load parcel data

Self-contained paginated download from the NJ statewide Cadastral MapServer
(`maxRecordCount = 1000`), filtered to consolidated Princeton (`PCL_MUN = '1114'`),
output in EPSG:4326. The **full MOD-IV attribute set** is pulled — including the
residential dwelling-unit count (`DWELL`), `COMM_DWELL`, `BLDG_CLASS`, `PROP_USE`,
`LAND_DESC`, and the sale fields (`SALE_PRICE`, `SALES_CODE`, `DEED_DATE`) — not
just the 16 value/geometry fields of the earlier pull. ESRI rings are converted to
shapely geometry (largest ring = exterior; counter-clockwise rings = holes).
Cached to `data/parcels.gpq`.

In [ ]:
PARCEL_PATH = DATA_DIR / 'parcels.gpq'
QUERY_URL = 'https://maps.nj.gov/arcgis/rest/services/Framework/Cadastral/MapServer/0/query'
WHERE = "PCL_MUN='1114'"     # consolidated Princeton (equivalent to legacy CD_CODE='1114', + null-class line items)
# Full MOD-IV attribute pull: values/geometry + the residential-unit, building,
# use, and sale fields needed for the richer NJ category logic (Section 3).
OUT_FIELDS = (
    'OBJECTID,PAMS_PIN,PCLQCODE,PCL_MUN,CD_CODE,COUNTY,MUN_NAME,PROP_CLASS,PROP_LOC,'
    'PROP_USE,LAND_DESC,BLDG_CLASS,DWELL,COMM_DWELL,'
    'LAND_VAL,IMPRVT_VAL,NET_VALUE,LAST_YR_TX,'
    'SALE_PRICE,SALES_CODE,DEED_DATE,CALC_ACRE,YR_CONSTR'
)


def esri_rings_to_geom(rings):
    if not rings:
        return None
    valid = [r for r in rings if r and len(r) >= 4]
    if not valid:
        return None
    if len(valid) == 1:
        g = Polygon(valid[0])
    else:
        areas = [abs(LinearRing(r).area) for r in valid]
        ext = valid[int(np.argmax(areas))]
        holes = [r for r in valid if r is not ext and LinearRing(r).is_ccw]
        try:
            g = Polygon(ext, holes)
        except Exception:
            g = Polygon(ext)
    if not g.is_valid:
        g = g.buffer(0)
    return g


if PARCEL_PATH.exists():
    gdf = gpd.read_parquet(PARCEL_PATH)
    print(f"Loaded {len(gdf):,} parcels from cache")
else:
    total = requests.get(QUERY_URL, params={'f': 'json', 'where': WHERE,
                                            'returnCountOnly': 'true'}, timeout=90).json()['count']
    print(f"Total Princeton parcels to fetch: {total:,}")
    rows = []
    offset = 0
    chunk = 1000    # NJOGIS maxRecordCount
    while offset < total:
        params = {
            'f': 'json', 'where': WHERE, 'outFields': OUT_FIELDS,
            'returnGeometry': 'true', 'outSR': 4326, 'geometryPrecision': 6,
            'resultOffset': offset, 'resultRecordCount': chunk,
        }
        data = requests.get(QUERY_URL, params=params, timeout=180).json()
        feats = data.get('features', [])
        if not feats:
            break
        for f in feats:
            a = dict(f['attributes'])
            g = f.get('geometry')
            a['geometry'] = esri_rings_to_geom(g.get('rings')) if g else None
            rows.append(a)
        offset += len(feats)
        print(f"  fetched {offset:,}/{total:,}")
        if len(feats) < chunk:
            break
        time.sleep(0.15)
    gdf = gpd.GeoDataFrame(rows, geometry='geometry', crs='EPSG:4326')
    gdf.to_parquet(PARCEL_PATH)
    print(f"Downloaded and cached {len(gdf):,} parcels -> {PARCEL_PATH}")

print("Geometry valid fraction:", round(gdf.geometry.is_valid.mean(), 4))
print("Null geometry:", int(gdf.geometry.isna().sum()))
print("Total bounds:", gdf.total_bounds)
print("Fields pulled:", [c for c in gdf.columns if c != 'geometry'])


## Section 3 — Classify and validate

**Exemption:** NJ property classes **15A-15F** are fully exempt (public, school,
church/charitable, cemetery, other exempt — dominated by Princeton University).
These are held out of the revenue-neutral base and excluded from the export and
charts (preserving all existing exemptions, per scope).

**Non-real-property rows:** the `PCL_MUN='1114'` pull surfaces ~448 line items
with a **blank `PROP_CLASS`** and **$0** land/improvement/net value (business
personal property / administrative rows the legacy `CD_CODE` pull omitted).
They are dropped; because they carry no value the taxable base and revenue are
unchanged.

**Category map (NJ MOD-IV, richer taxonomy using `DWELL` + `PCLQCODE`):**

| Class / rule | Category |
|---|---|
| 1 | Vacant Land |
| 2, `PCLQCODE` starts with `C` | Condominium |
| 2, non-condo, `DWELL` = 1 (or null/0) | Single Family Residential |
| 2, non-condo, `DWELL` 2-4 | Small Multi-Family (2-4 units) |
| 4C | Large Multi-Family (5+ units) — apartments |
| 4A | Commercial |
| 4B | Industrial |
| 3A / 3B | Agricultural (farmland) |
| 15A-15F | Exempt (held out) |

This replaces the earlier coarse pull, which lumped all 1-4-unit class-2
residential into one "Single Family" bucket (hiding ~415 small-multifamily homes)
and buried the class-3 farm parcels via a `$0`-improvement→Vacant override. That
override is **removed** here: Vacant Land is class 1 only.

In [ ]:
# Numeric cleanup
for c in ['LAND_VAL', 'IMPRVT_VAL', 'NET_VALUE', 'LAST_YR_TX']:
    gdf[c] = pd.to_numeric(gdf[c], errors='coerce').fillna(0.0)
gdf['DWELL_N'] = pd.to_numeric(gdf['DWELL'], errors='coerce')     # residential dwelling-unit count

gdf['taxable_land_value'] = gdf['LAND_VAL'].clip(lower=0)
gdf['taxable_improvement_value'] = gdf['IMPRVT_VAL'].clip(lower=0)
gdf['taxable_total'] = gdf['taxable_land_value'] + gdf['taxable_improvement_value']

# Normalized class / condo-qualifier strings
pc = gdf['PROP_CLASS'].apply(lambda x: str(x).strip().upper() if pd.notna(x) else '')
qc = gdf['PCLQCODE'].apply(lambda x: str(x).strip().upper() if pd.notna(x) else '')

# Drop non-real-property line items: blank/null PROP_CLASS. The PCL_MUN pull
# surfaces ~448 such records (business personal property / administrative rows)
# that the legacy CD_CODE pull omitted. They carry ZERO land, improvement, and
# net value, so removing them leaves the taxable base (and revenue) unchanged.
nonreal = (pc == '')
assert gdf.loc[nonreal, 'taxable_total'].sum() == 0, "null-class rows carry value — investigate"
print(f"Dropping {int(nonreal.sum())} null-PROP_CLASS line items (all $0 value; not real property)")
gdf = gdf[~nonreal].copy()
pc = pc[~nonreal]
qc = qc[~nonreal]

# Exemption flag: NJ class 15A-15F fully exempt (dominated by Princeton University)
gdf['full_exmp'] = pc.str.startswith('15').astype(int)


def classify(c, q, dwell):
    """NJ MOD-IV property category (canonical cross-city taxonomy names).

    15A-15F -> Exempt (held out). Class 1 -> Vacant Land. Class 2 splits by
    condo qualifier and dwelling-unit count:
      - C-prefixed PCLQCODE                 -> Condominium
      - DWELL == 1 (or null / 0, defaulted) -> Single Family Residential
      - DWELL 2-4                           -> Small Multi-Family (2-4 units)
    Class 4C -> Large Multi-Family (5+ units) [apartments]; 4A -> Commercial;
    4B -> Industrial; 3A/3B -> Agricultural [farmland].
    """
    if c.startswith('15'):
        return 'Exempt'
    if c == '1':
        return 'Vacant Land'
    if c == '2':
        if q.startswith('C'):
            return 'Condominium'
        if pd.notna(dwell) and 2 <= dwell <= 4:
            return 'Small Multi-Family (2-4 units)'
        return 'Single Family Residential'   # DWELL==1, or null/0 -> default SF
    if c == '4C':
        return 'Large Multi-Family (5+ units)'
    if c == '4A':
        return 'Commercial'
    if c == '4B':
        return 'Industrial'
    if c in ('3A', '3B'):
        return 'Agricultural'
    return 'Other'


gdf['PROPERTY_CATEGORY'] = [classify(pc[i], qc[i], gdf.loc[i, 'DWELL_N']) for i in gdf.index]

print("Total real-property parcels:", len(gdf))
print("Fully exempt (class 15x):", int(gdf['full_exmp'].sum()))
print("\nCategory counts (all real-property parcels):")
print(gdf['PROPERTY_CATEGORY'].value_counts())

# Base reconciliation vs certified ratable base
taxable = gdf[gdf['full_exmp'] == 0]
print(f"\nModeled taxable parcels: {len(taxable):,}")
print(f"Taxable assessed base:   ${taxable['taxable_total'].sum():,.0f}")
print(f"Certified net val (assd):${CERTIFIED_NET_VAL:,.0f}")
print(f"Base coverage:           {taxable['taxable_total'].sum()/CERTIFIED_NET_VAL*100:.2f}% "
      f"(shortfall = business personal property + minor missing parcels, not real property)")


## Section 4 — Current municipal tax and revenue gate

Current municipal tax per parcel = assessed total x municipal millage / 1000,
where the municipal millage is the **certified municipal (purpose + library)
rate** = official levy / certified net valuation taxable. Fully-exempt parcels
pay $0. We validate the modeled municipal revenue against the official
**$40,100,104** levy (Gate 1).

In [ ]:
gdf['millage_rate'] = MUNI_MILLAGE

current_revenue, _, gdf = calculate_current_tax(
    df=gdf,
    tax_value_col='taxable_total',
    millage_rate_col='millage_rate',
    exemption_flag_col='full_exmp',
)

gap_pct = (current_revenue / OFFICIAL_REVENUE - 1) * 100
print(f"Modeled municipal revenue: ${current_revenue:,.0f}")
print(f"Official municipal levy:   ${OFFICIAL_REVENUE:,.0f}")
print(f"Gap:                       {gap_pct:+.2f}%")
assert abs(gap_pct) < 5.0, f"Revenue gap {gap_pct:.1f}% exceeds 5% threshold"
print("Gate 1 PASS (gap reflects parcel base vs certified ratable base; see Section 3).")


## Section 5 — Split-rate model (2:1)

Fully-exempt parcels are held out of the solver and dropped from the modeled
output (they carry no signal: current = new = $0). The 2:1 split-rate is solved
revenue-neutral against the modeled municipal current tax on the remaining
taxable parcels.

In [ ]:
n_exempt = int((gdf['full_exmp'] == 1).sum())
gdf = gdf[gdf['full_exmp'] == 0].copy()

land_millage, improvement_millage, new_revenue, gdf = model_split_rate_tax(
    df=gdf,
    land_value_col='taxable_land_value',
    improvement_value_col='taxable_improvement_value',
    current_revenue=gdf['current_tax'].sum(),
    land_improvement_ratio=LAND_IMPROVEMENT_RATIO,
)

print(f"Held out {n_exempt:,} fully-exempt parcels (excluded from model, export, charts).")
print(f"Land millage:        {land_millage:.4f}  (${land_millage/10:.4f}/$100)")
print(f"Improvement millage: {improvement_millage:.4f}  (${improvement_millage/10:.4f}/$100)")
print(f"Revenue-neutral check: ${new_revenue:,.0f}  vs current ${gdf['current_tax'].sum():,.0f}")

category_summary = calculate_category_tax_summary(
    df=gdf,
    category_col='PROPERTY_CATEGORY',
    current_tax_col='current_tax',
    new_tax_col='new_tax',
)
print_category_tax_summary(category_summary, title=f"{CITY_NAME} — 2:1 Split-Rate Tax Impact")

### Gate 5 — Artifact scan

Read the category table against economic priors: building-heavy categories
(commercial, apartments, condos) should see building shares of ~20-60% and
tax *decreases*; land-heavy categories (vacant land) should see large
*increases*. Watch for distinct categories pinned at an identical extreme
(placeholder building values) — NJ MOD-IV gives condos real land/improvement
splits, so the St. Paul token-condo artifact should not appear here.

In [ ]:
d = gdf.copy()
d['bldg_share'] = d['taxable_improvement_value'] / (
    d['taxable_land_value'] + d['taxable_improvement_value']).replace(0, np.nan)
scan = d.groupby('PROPERTY_CATEGORY').agg(
    n=('tax_change_pct', 'size'),
    median_pct=('tax_change_pct', 'median'),
    median_bldg_share=('bldg_share', 'median'),
    pct_zero_bldg=('taxable_improvement_value', lambda s: (s <= 1000).mean()),
).sort_values('median_pct', ascending=False)
print(scan.round(3).to_string())


## Section 6 — Census join + standard export + report + parcel map

Closing pattern per the build-notebook skill. NJ FIPS 34 + Mercer 021 = 34021.

In [ ]:
import concurrent.futures

_fips = STATE_FIPS + COUNTY_FIPS
try:
    with concurrent.futures.ThreadPoolExecutor(max_workers=1) as _ex:
        _future = _ex.submit(get_census_data_with_boundaries, _fips, 2022)
        try:
            census_data, census_gdf = _future.result(timeout=90)
            gdf = match_to_census_blockgroups(gdf, census_gdf)
            if 'minority_pct' not in gdf.columns and 'total_pop' in gdf.columns and 'white_pop' in gdf.columns:
                gdf['minority_pct'] = ((gdf['total_pop'] - gdf['white_pop']) / gdf['total_pop'] * 100).round(2)
            if 'black_pct' not in gdf.columns and 'total_pop' in gdf.columns and 'black_pop' in gdf.columns:
                gdf['black_pct'] = (gdf['black_pop'] / gdf['total_pop'] * 100).round(2)
            print(f"Census join: {gdf['std_geoid'].notna().mean()*100:.1f}% matched")
        except concurrent.futures.TimeoutError:
            print("Census API timed out — skipping census join")
            for _col in ['std_geoid', 'median_income', 'minority_pct', 'black_pct']:
                gdf[_col] = float('nan')
except Exception as e:
    print(f"Census join failed: {e}")
    for _col in ['std_geoid', 'median_income', 'minority_pct', 'black_pct']:
        gdf[_col] = float('nan')


In [ ]:
out_df = save_standard_export(
    df=gdf,
    city=CITY_NAME,
    output_path=f'../../analysis/data/{CITY_NAME}.csv',
    model_type=MODEL_TYPE,
    land_millage=land_millage,
    improvement_millage=improvement_millage,
    property_category_col='PROPERTY_CATEGORY',
    current_tax_col='current_tax',
    new_tax_col='new_tax',
    tax_change_col='tax_change',
    tax_change_pct_col='tax_change_pct',
    taxable_land_col='taxable_land_value',
    taxable_improvement_col='taxable_improvement_value',
    exempt_flag_col='full_exmp',
)

# Category names use the canonical cross-city taxonomy, so the report's default
# residential quintile filter (_RESIDENTIAL_CATEGORIES) already covers Single
# Family / Small & Large Multi-Family / Condominium.
create_city_report(out_df, CITY_NAME, show=False)

map_gdf = save_parcel_map_export(
    gdf=gdf,
    city=CITY_NAME,
    output_path=f'../../analysis/maps/{CITY_NAME}.parquet',
    model_type=MODEL_TYPE,
    land_millage=land_millage,
    improvement_millage=improvement_millage,
    parcel_id_col=PARCEL_ID_COL,
    parcel_url_template=PARCEL_URL_TEMPLATE,
    owner_name_col=OWNER_NAME_COL,
    owner_address_col=OWNER_ADDRESS_COL,
)
create_parcel_map(map_gdf, CITY_NAME)
print("Done.")


## Section 7 — Single-Family incidence by home value quartile

An honest incidence view: split the **Single Family Residential** parcels
(DWELL = 1, non-condo) into quartiles by **assessed total value** and report the
median tax change and the share getting a cut in each quartile. This isolates
whether the 2:1 shift is progressive or regressive *among homeowners*.

The pattern is regressive and **real** (not an assessment artifact): NJ MOD-IV
assessment-to-sale ratios are roughly flat (~0.70) across the value range in
Princeton, so cheaper homes — which sit on a higher land-to-improvement share —
carry the increase, while the priciest homes (bigger structures on their lots)
get the cut. Saved as `sf_value_quartile_incidence.png`.

In [ ]:
import os

sf = out_df[out_df['property_category'] == 'Single Family Residential'].copy()
sf['total_value'] = sf['taxable_land_value'] + sf['taxable_improvement_value']
sf = sf[sf['total_value'] > 0]

Q_LABELS = ['Q1 (cheapest)', 'Q2', 'Q3', 'Q4 (priciest)']
sf['value_quartile'] = pd.qcut(sf['total_value'], 4, labels=Q_LABELS)

sf_quartile = sf.groupby('value_quartile', observed=True).agg(
    n=('tax_change_pct', 'size'),
    value_low=('total_value', 'min'),
    value_high=('total_value', 'max'),
    median_pct=('tax_change_pct', 'median'),
    share_cut=('tax_change_pct', lambda s: float((s < 0).mean())),
).reset_index()

print(f"Single Family Residential: {len(sf):,} parcels; "
      f"overall median {sf['tax_change_pct'].median():+.2f}%, "
      f"share getting a cut {(sf['tax_change_pct'] < 0).mean()*100:.1f}%\n")
print(sf_quartile.assign(
    median_pct=lambda d: d['median_pct'].round(2),
    share_cut=lambda d: (d['share_cut']*100).round(1),
).to_string(index=False))

# Figure: median % change by value quartile, colored by direction, annotated with % getting a cut
fig, ax = plt.subplots(figsize=(8, 5))
colors = ['#B3261E' if v > 0 else '#2E7D32' for v in sf_quartile['median_pct']]
bars = ax.bar(sf_quartile['value_quartile'].astype(str), sf_quartile['median_pct'],
              color=colors, edgecolor='white', width=0.7)
ax.axhline(0, color='#333', linewidth=0.8)
ax.set_ylabel('Median tax change (%)')
ax.set_title('Princeton 2:1 Split-Rate — Single-Family Incidence by Home Value Quartile',
             fontsize=12, fontweight='bold')
for bar, (_, r) in zip(bars, sf_quartile.iterrows()):
    y = bar.get_height()
    va = 'bottom' if y >= 0 else 'top'
    off = 0.4 if y >= 0 else -0.4
    ax.text(bar.get_x() + bar.get_width()/2, y + off,
            f"{y:+.1f}%\n{r['share_cut']*100:.0f}% get a cut",
            ha='center', va=va, fontsize=9)
ax.margins(y=0.18)
ax.spines[['top', 'right']].set_visible(False)
fig.tight_layout()
_p = os.path.join('../../analysis/reports', CITY_NAME, 'sf_value_quartile_incidence.png')
fig.savefig(_p, dpi=150, bbox_inches='tight')
plt.close(fig)
print(f"\nSaved {_p}")


## Validation Summary

| Check | Result |
|---|---|
| Revenue match (Gate 1) | Modeled municipal revenue vs official $40,100,104 (Municipal Purpose + Library, 2024 Mercer Abstract of Ratables) |
| Scope | City/municipal levy only — excl. county, school, and municipal open-space dedicated tax |
| Parcel count | ~7,634 modeled taxable + 757 fully-exempt (class 15x) held out + ~448 null-class $0 line items dropped |
| Categories | Richer NJ taxonomy: Single Family (DWELL=1), Small Multi-Family (DWELL 2-4), Large Multi-Family (4C), Condominium (class-2 C-qualifier), Vacant Land (class 1), Commercial (4A), Industrial (4B), Agricultural/farmland (3A/3B) |
| Assessment | Director's ratio 70.51% (cancels; modeled on assessed values) |
| Legality note | NJ does not currently authorize municipal split-rate/LVT; this is a hypothetical revenue-neutral model (legal framing handled downstream) |

**Data source:** Parcels and MOD-IV Composite of New Jersey (NJOGIS),
`maps.nj.gov/.../Framework/Cadastral/MapServer/0`, filtered `PCL_MUN='1114'`
(full attribute set incl. `DWELL`/`PCLQCODE` for the category logic).